##### Copyright 2026 Google LLC.

In [ ]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Illustrating a book using Nano Banana and GenMedia models

<a target="_blank" href="https://colab.research.google.com/github/google-gemini/cookbook/blob/main/examples/Book_illustration.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" height=30/></a>

In this guide, you will pair Gemini's core capabilities (long context, native multimodality, structured outputs, the Files API, and stateful interaction chaining) with **[Nano Banana](../quickstarts/Get_Started_Nano_Banana.ipynb)** to illustrate both a text book and an audiobook with consistent characters.

You will also bring your illustrations to life across video, music, and narration using Google DeepMind's generative media models:
- 🎬 **[Gemini Omni 1.2 Flash](../quickstarts/Get_started_Omni.ipynb)** — Animate a portrait chapter illustration into a story-continuation action video with synchronized sound effects
- 🎵 **[Lyria 3.5](../quickstarts/Get_started_Lyria.ipynb)** — Compose an instrumental background soundtrack tailored to each chapter
- 🗣️ **[Gemini 3.8 TTS](../quickstarts/Get_started_TTS.ipynb)** — Perform a multi-character theatrical scene across two voices using per-character vocal persona prompts

Each concept is explained step by step. For a focused introduction to image generation on its own, check out the [Nano Banana quickstart](../quickstarts/Get_Started_Nano_Banana.ipynb) or the [Image generation documentation](https://ai.google.dev/gemini-api/docs/image-generation).

Note: Prompts are extracted for every chapter in the book, while image and soundtrack rendering default to 3 character portraits and 3 chapters to keep notebook size and API usage compact—raise those sliders anytime to render more.

> **Note:** [Enable billing](https://ai.google.dev/gemini-api/docs/billing#enable-cloud-billing) to use paid image and GenMedia models (see [pricing](https://ai.google.dev/pricing)), or select `gemini-3.1-flash-lite-image` (or `gemini-2.5-flash-image`) for free-tier image generation.
>
> This notebook also includes optional paid sections for **Video generation** ([Gemini Omni 1.2 Flash](../quickstarts/Get_started_Omni.ipynb)), **Music generation** ([Lyria 3.5](../quickstarts/Get_started_Lyria.ipynb)), and **Text-to-speech** ([Gemini 3.8 TTS](../quickstarts/Get_started_TTS.ipynb)). Toggle `I_understand_this_is_a_paid_API` to `True` in Setup before running them.

## 0/ Setup

This section installs the `google-genai` SDK (`>=3.2.0`), loads your [API key](../quickstarts/Authentication.ipynb), initializes the client, and selects the models used throughout the notebook.

Collapse and run this section to jump straight to the examples.

### Install SDK

In [ ]:
%pip install -U -q "google-genai>=3.2.0"

### Set up your API key

To run the following cell, store your API key in a Colab Secret named `GEMINI_API_KEY`. If you do not already have an API key or need help creating a Colab Secret, see [Authentication ![image](https://storage.googleapis.com/generativeai-downloads/images/colab_icon16.png)](../quickstarts/Authentication.ipynb).

In [ ]:
from google.colab import userdata

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

### Initialize SDK client

Initialize `genai.Client` with your API key and exponential-backoff `retry_options` for transient `429` and `5xx` responses (or use OAuth if calling [Vertex AI](https://cloud.google.com/vertex-ai/generative-ai/docs/overview)).

In [ ]:
from google import genai
from google.genai import types

retry_options = types.HttpRetryOptions(
    attempts=5,
    initial_delay=2.0,
    max_delay=60.0,
    http_status_codes=[429, 500, 502, 503, 504],
)

client = genai.Client(
    api_key=GEMINI_API_KEY,
    http_options=types.HttpOptions(retry_options=retry_options),
)

### Helper functions

Run this hidden cell to define display and Files API polling helpers reused across the notebook.

In [ ]:
# @title Helper functions { display-mode: "form" }
import base64
import html
import time
from IPython.display import Image as IPyImage
from IPython.display import Markdown, display


def safe_md(text: object) -> Markdown:
    """Escape model text before rendering as Markdown."""
    return Markdown(html.escape(str(text or ""), quote=True))


def display_generated_image(img_obj: object, width: int = 384) -> bytes:
    """Decode and display a generated interaction image inline."""
    img_bytes = base64.b64decode(img_obj.data)
    display(IPyImage(data=img_bytes, width=width))
    return img_bytes


def wait_for_file_active(
    file_obj: object, file_client: object = None, max_polls: int = 30
) -> object:
    """Poll a Files API upload until its state becomes ACTIVE."""
    active_client = file_client or client
    for _ in range(max_polls):
        state_name = getattr(file_obj.state, "name", file_obj.state)
        if state_name == "ACTIVE":
            return file_obj
        if state_name == "FAILED":
            raise RuntimeError(f"File upload failed: {file_obj.name}")
        time.sleep(2)
        file_obj = active_client.files.get(name=file_obj.name)
    raise TimeoutError(
        f"Timed out waiting for {file_obj.name} to become ACTIVE"
    )

### Select models and inference tier

Choose the models for prompt writing, image generation, music, and narration, and confirm `I_understand_this_is_a_paid_API` if you want to run the paid video, music, and TTS sections.

You can also choose a `service_tier` for supported text and image calls (`GEMINI_MODEL_ID` and `IMAGE_MODEL_ID`):
- **`standard`** (default): Standard pay-as-you-go pricing and latency.
- **[`priority`](https://ai.google.dev/gemini-api/docs/priority-inference)**: Routes requests to a dedicated low-latency queue at higher per-token pricing, with graceful fallback to `"standard"` if a burst exceeds your priority pool.
- **`flex`**: Lower-cost batch-oriented tier for latency-tolerant workloads (see the [Inference tiers quickstart](../quickstarts/Inference_tiers.ipynb)).

In [ ]:
IMAGE_MODEL_ID = "gemini-nano-banana-2.1"  # @param ["gemini-nano-banana-2.1", "gemini-3-pro-image-preview", "gemini-3.1-flash-image", "gemini-3.1-flash-lite-image", "gemini-2.5-flash-image"] {"allow-input": true, "isTemplate": true}
GEMINI_MODEL_ID = "gemini-3.8-flash"  # @param ["gemini-4-carbon", "gemini-4-argon", "gemini-4-neon", "gemini-4-hydrogen", "gemini-3.1-pro-preview", "gemini-3.8-flash", "gemini-3.7-flash", "gemini-3.6-flash", "gemini-3.5-flash-lite", "gemini-2.5-pro"] {"allow-input": true, "isTemplate": true}

# Models for optional paid music and narration sections
LYRIA_MODEL_ID = "lyria-3.5"  # @param ["lyria-3-pro-preview", "lyria-3.5", "lyria-3-clip-preview"] {"allow-input": true, "isTemplate": true}
TTS_MODEL_ID = "gemini-3.8-flash-tts"  # @param ["gemini-3.8-flash-tts", "gemini-3.8-flash-lite-tts"] {"allow-input": true, "isTemplate": true}

# Toggle paid GenMedia sections and select your inference service tier
I_understand_this_is_a_paid_API = False  # @param {"type": "boolean"}
service_tier = "standard"  # @param ["flex", "standard", "priority"]

To keep notebook output compact and control token usage, generation defaults to 3 character portraits and 3 chapters—adjust either value below if you want more.

In [ ]:
max_character_images = 3  # @param {"type": "integer", "isTemplate": true, "min": 1}
max_chapter_images = 3  # @param {"type": "integer", "isTemplate": true, "min": 1}

# Illustrate a book: The Wind in the Willows

## 1/ Get a book and upload using the Files API

Download a public-domain book—defaulting to the [Internet Archive mirror](https://archive.org/details/thewindinthewill00289gut) of Project Gutenberg's *[The Wind in the Willows](https://en.wikipedia.org/wiki/The_Wind_in_the_Willows)* by Kenneth Grahame (with Project Gutenberg URLs also available in the dropdown, or paste any plain-text `.txt` URL)—and upload it with the Files API (`client.files.upload`).

In [ ]:
import requests

url = "https://archive.org/download/thewindinthewill00289gut/wwill11.txt"  # @param ["https://archive.org/download/thewindinthewill00289gut/wwill11.txt", "https://www.gutenberg.org/cache/epub/289/pg289.txt", "https://www.gutenberg.org/cache/epub/37472/pg37472.txt"] {"allow-input": true}

response = requests.get(
    url,
    headers={"User-Agent": "Mozilla/5.0"},
    timeout=30,
)
response.raise_for_status()
text = response.text

# Strip Project Gutenberg license footer if present
end_marker = "*** END OF THE PROJECT GUTENBERG EBOOK"
if end_marker in text:
    text = text[: text.index(end_marker)]

with open("book.txt", "w", encoding="utf-8") as file:
    file.write(text)

book = wait_for_file_active(client.files.upload(file="book.txt"))
print(f"Uploaded {book.name} ({len(text.encode('utf-8')):,} bytes)")

Define shared visual rules (`system_instructions`) telling the image model to produce clean, borderless illustrations with no embedded text or multi-panel layouts:

In [ ]:
system_instructions = """
    There must be no text on the image, and it should not look like a cover page.
    It should be a full illustration with no borders, titles, or captions.
    Unless asked otherwise, stay family-friendly with uplifting colors.
    Each output should be a single full-bleed scene with no comic panels.
"""

## 2/ Start the chat

Chain interactions via `previous_interaction_id` so Gemini retains conversation history and the uploaded book across turns without resending the file. See the [Get started notebook](../quickstarts/Get_started.ipynb) for more on stateful interactions.

Use [structured outputs](https://ai.google.dev/gemini-api/docs/structured-output?lang=python#generate-json) with a Pydantic `Prompt` schema so Gemini returns clean JSON lists of character and chapter prompts:

In [ ]:
from pydantic import BaseModel


class Prompt(BaseModel):
    name: str
    prompt: str

`client.interactions.create` starts the stateful session with the uploaded book and reports which `service_tier` served the request:

In [ ]:
# Start the conversation with the uploaded book
book_interaction = client.interactions.create(
    model=GEMINI_MODEL_ID,
    input=[
        {
            "type": "text",
            "text": (
                "Here is a book to illustrate using Nano Banana. "
                "Do not say anything for now; instructions will follow."
            ),
        },
        {"type": "document", "uri": book.uri},
    ],
    service_tier=service_tier,
)
print(
    f"Session started: {book_interaction.id} "
    f"(tier: {book_interaction.service_tier})"
)

Sending the book in its own initial turn anchors the full manuscript in the conversation state so you can branch or iterate on styles and prompts without re-uploading the file.

## 3/ Define a style

Type a custom visual direction in `style` below, or leave `style` empty to let Gemini invent an art style tailored to the story (with a creative twist).

In [ ]:
style = ""  # @param {"type": "string", "placeholder": "Write your own style or leave empty to let Gemini generate one"}

if style == "":
    style_interaction = client.interactions.create(
        model=GEMINI_MODEL_ID,
        input=(
            "Define an art style that fits this story with an unexpected "
            "creative twist. Return only the concise style description to "
            "append to future image generation prompts."
        ),
        previous_interaction_id=book_interaction.id,
        service_tier=service_tier,
    )
    last_interaction = style_interaction
    style = style_interaction.output_text
else:
    style_interaction = client.interactions.create(
        model=GEMINI_MODEL_ID,
        input=(
            f'The art style will be: "{style}". Keep that in mind when '
            "generating future prompts. Wait quietly for the next instructions."
        ),
        previous_interaction_id=book_interaction.id,
        service_tier=service_tier,
    )
    last_interaction = style_interaction

display(Markdown("### Style"))
print(style)

style = f'Follow this style: "{style}"'

## 4/ Generate portraits of the main characters

Ask Gemini to identify the main recurring animal and adult characters across the book (excluding human children for regional image generation compatibility) and craft detailed visual descriptions (at least 50 words each) adhering to the `Prompt` JSON schema:

In [ ]:
import json

characters_prompts_interaction = client.interactions.create(
    model=GEMINI_MODEL_ID,
    input=(
        "Identify the main recurring animal or adult characters across the "
        "entire book (do not include human children) and write a richly "
        "detailed portrait prompt (at least 50 words each, grounded in the "
        "text) so Nano Banana can illustrate each character."
    ),
    previous_interaction_id=style_interaction.id,
    response_format={
        "type": "text",
        "mime_type": "application/json",
        "schema": {"type": "array", "items": Prompt.model_json_schema()},
    },
    service_tier=service_tier,
)
last_interaction = characters_prompts_interaction

all_characters = json.loads(characters_prompts_interaction.output_text)
print(f"{len(all_characters)} character prompts ready.")
print(json.dumps(all_characters, indent=2))

Loop over the character prompts and generate portrait illustrations with **Nano Banana** (`IMAGE_MODEL_ID`). Chaining each turn via `previous_interaction_id` keeps palette, brushwork, and proportions cohesive across characters.

For a deeper dive into Nano Banana options, see the [Nano Banana quickstart](../quickstarts/Get_Started_Nano_Banana.ipynb). Key parameters used below:
* `response_format={"type": "image", "aspect_ratio": "9:16"}` requests vertical portrait illustrations directly.
* `system_instruction=system_instructions` enforces borderless artwork without text overlays on every turn.
* `interaction.output_image` retrieves the generated image payload (`data` and `mime_type`) directly.

In [ ]:
characters = all_characters[:max_character_images]
print(
    f"Rendering {len(characters)} of {len(all_characters)} "
    "character portraits..."
)
character_images = []
last_image_interaction = None

for character in characters:
    display(safe_md(f"### {character['name']}"))
    display(safe_md(character["prompt"]))

    turn_kwargs = {
        "model": IMAGE_MODEL_ID,
        "input": (
            "Illustrate a character portrait for this storybook.\n"
            f"{style}\n"
            f"Character ({character['name']}): {character['prompt']}"
        ),
        "system_instruction": system_instructions,
        "response_format": {"type": "image", "aspect_ratio": "9:16"},
        "service_tier": service_tier,
    }
    if last_image_interaction is not None:
        turn_kwargs["previous_interaction_id"] = last_image_interaction.id

    characters_image_interaction = client.interactions.create(**turn_kwargs)
    last_image_interaction = characters_image_interaction

    generated_image = characters_image_interaction.output_image
    if generated_image and generated_image.data:
        display_generated_image(generated_image)
    else:
        print(f"No image generated for {character['name']}")

    character_images.append(generated_image)

## 5/ Illustrate the chapters of the book

Ask Gemini to extract a single-scene illustration prompt for **every** chapter in the book in one structured JSON call (`all_chapters`), then slice `all_chapters[:max_chapter_images]` in Python when rendering images so you can adjust `max_chapter_images` without re-extracting prompts:

In [ ]:
chapters_prompts_interaction = client.interactions.create(
    model=GEMINI_MODEL_ID,
    input=(
        "Write one detailed single-scene illustration prompt for EVERY "
        "chapter or story in the book in chronological order (cover all "
        "chapters from first to last; no multi-panel pages). Name every "
        "character appearing in each scene and reuse their physical "
        "descriptions from earlier."
    ),
    previous_interaction_id=characters_prompts_interaction.id,
    response_format={
        "type": "text",
        "mime_type": "application/json",
        "schema": {"type": "array", "items": Prompt.model_json_schema()},
    },
    service_tier=service_tier,
)
last_interaction = chapters_prompts_interaction

all_chapters = json.loads(chapters_prompts_interaction.output_text)
print(f"{len(all_chapters)} chapters ready.")
print(json.dumps(all_chapters, indent=2))

In [ ]:
import io
from PIL import Image

chapters = all_chapters[:max_chapter_images]
print(f"Illustrating {len(chapters)} of {len(all_chapters)} chapters...")
chapter_illustrations = []
last_chapter_image = None

for chapter in chapters:
    display(safe_md(f"### {chapter['name']}"))
    display(safe_md(chapter["prompt"]))

    turn_kwargs = {
        "model": IMAGE_MODEL_ID,
        "input": (
            f"Create a chapter illustration for {chapter['name']} keeping "
            "the previously generated characters visually consistent while "
            f"adapting their poses to the scene:\n{style}\n{chapter['prompt']}"
        ),
        "system_instruction": system_instructions,
        "response_format": {"type": "image", "aspect_ratio": "9:16"},
        "service_tier": service_tier,
    }
    if last_image_interaction is not None:
        turn_kwargs["previous_interaction_id"] = last_image_interaction.id

    chapters_image_interaction = client.interactions.create(**turn_kwargs)
    last_image_interaction = chapters_image_interaction

    generated_image = chapters_image_interaction.output_image
    if generated_image and generated_image.data:
        img_bytes = display_generated_image(generated_image)
        pil_image = Image.open(io.BytesIO(img_bytes))
        last_chapter_image = pil_image
        chapter_illustrations.append({"chapter": chapter, "image": pil_image})

### Bonus: Going further with more granular control

When a book has a large cast of characters (or recurring locations and props), chaining dozens of turns can accumulate unnecessary context. Instead, ask Gemini to list the exact characters appearing in each chapter (`all_bonus_chapters`), slice to `max_chapter_images` in Python, and pass **only** those character portraits as explicit multi-image references on a fresh interaction turn:

In [ ]:
class Chapter(BaseModel):
    name: str
    prompt: str
    characters: list[str]

In [ ]:
bonus_prompts_interaction = client.interactions.create(
    model=GEMINI_MODEL_ID,
    input=(
        "Write one detailed single-scene illustration prompt for EVERY "
        "chapter or story in the book in chronological order (cover all "
        "chapters from first to last), reusing the character descriptions, "
        "and list the exact character names from your earlier character list "
        "who appear in each scene."
    ),
    previous_interaction_id=characters_prompts_interaction.id,
    response_format={
        "type": "text",
        "mime_type": "application/json",
        "schema": {"type": "array", "items": Chapter.model_json_schema()},
    },
    service_tier=service_tier,
)
last_interaction = bonus_prompts_interaction

all_bonus_chapters = json.loads(bonus_prompts_interaction.output_text)
print(f"{len(all_bonus_chapters)} chapters ready.")
print(json.dumps(all_bonus_chapters, indent=2))

Generate each chapter scene in a standalone call by attaching only the matching character reference portraits:

In [ ]:
character_lookup = {
    char_data["name"]: character_images[i]
    for i, char_data in enumerate(characters[: len(character_images)])
    if character_images[i] is not None
}
bonus_chapters = all_bonus_chapters[:max_chapter_images]
print(
    f"Illustrating {len(bonus_chapters)} of {len(all_bonus_chapters)} "
    "chapters with explicit character references..."
)
bonus_illustrations = []

for chapter in bonus_chapters:
    display(safe_md(f"### {chapter['name']}"))
    display(safe_md(chapter["prompt"]))

    image_inputs = [
        {
            "type": "image",
            "data": character_lookup[char_name].data,
            "mime_type": character_lookup[char_name].mime_type,
        }
        for char_name in chapter["characters"]
        if char_name in character_lookup
    ]

    interaction = client.interactions.create(
        model=IMAGE_MODEL_ID,
        input=[
            {
                "type": "text",
                "text": (
                    f"Create this illustration for {chapter['name']}:\n"
                    f"{style}\n{chapter['prompt']}\n"
                    "Use the provided images as character appearance "
                    "references."
                ),
            },
            *image_inputs,
        ],
        system_instruction=system_instructions,
        response_format={"type": "image", "aspect_ratio": "9:16"},
        service_tier=service_tier,
    )

    generated_image = interaction.output_image
    if generated_image and generated_image.data:
        img_bytes = display_generated_image(generated_image)
        pil_image = Image.open(io.BytesIO(img_bytes))
        last_chapter_image = pil_image
        bonus_illustrations.append({"chapter": chapter, "image": pil_image})

if bonus_illustrations:
    chapters = bonus_chapters
    chapter_illustrations = bonus_illustrations

## 6/ Animate a chapter illustration with Gemini Omni 1.2 Flash

Bring a rendered portrait chapter illustration to life with **[Gemini Omni 1.2 Flash](../quickstarts/Get_started_Omni.ipynb)** (`gemini-omni-1.2-flash`), Google DeepMind's native multimodal video generation and editing model built directly into the Interactions API (see the [Omni documentation](https://ai.google.dev/gemini-api/docs/omni)). Use `video_chapter_index` (`-1` for the latest illustrated chapter, or `0`, `1`, ...) and `video_duration_seconds` below to choose which scene to animate.

> **Note:** Video generation is a paid feature—check the [pricing page](https://ai.google.dev/pricing) for per-second rates.

In [ ]:
OMNI_MODEL_ID = "gemini-omni-1.2-flash"  # @param ["gemini-omni-1.2-flash", "gemini-omni-1.1-flash"] {"allow-input": true, "isTemplate": true}
video_chapter_index = -1  # @param {"type": "integer", "isTemplate": true}
video_duration_seconds = 8  # @param {"type": "slider", "min": 4, "max": 15, "step": 1}

Save the selected chapter illustration to `chapter_first_frame.png`, ask Gemini (using `previous_interaction_id` and the frame image) to choreograph what physically happens **next** in the story, then generate the video with `OMNI_MODEL_ID`. Because `google-genai>=3.2.0` defaults to `v1` while `gemini-omni-1.2-flash` is currently served on `v1beta`, this section initializes a scoped `v1beta` client with a 10-minute (`600000` ms) HTTP timeout for video generation:

In [ ]:
if not I_understand_this_is_a_paid_API:
    print(
        "Video generation is a paid feature. Set "
        "'I_understand_this_is_a_paid_API' to True above if you want to run it."
    )
else:
    from IPython.display import Video

    omni_client = genai.Client(
        api_key=GEMINI_API_KEY,
        http_options=types.HttpOptions(
            api_version="v1beta",
            timeout=600000,
            retry_options=retry_options,
        ),
    )

    if "chapter_illustrations" in locals() and chapter_illustrations:
        if (
            -len(chapter_illustrations)
            <= video_chapter_index
            < len(chapter_illustrations)
        ):
            selected_entry = chapter_illustrations[video_chapter_index]
        else:
            selected_entry = chapter_illustrations[-1]
        target_chapter = selected_entry["chapter"]
        target_frame_image = selected_entry["image"]
    else:
        target_chapter = chapters[-1]
        target_frame_image = globals().get("last_chapter_image")

    display(safe_md(f"### Animating: {target_chapter['name']}"))

    if target_frame_image is None:
        raise RuntimeError(
            "No chapter illustration available; run Section 5 first."
        )

    frame_path = "chapter_first_frame.png"
    target_frame_image.save(frame_path)
    with open(frame_path, "rb") as frame_file:
        frame_b64 = base64.b64encode(frame_file.read()).decode("utf-8")

    uploaded_frame = wait_for_file_active(
        omni_client.files.upload(file=frame_path),
        file_client=omni_client,
    )

    beat_1 = max(2, video_duration_seconds // 3)
    beat_2 = max(beat_1 + 1, (2 * video_duration_seconds) // 3)

    motion_prompt_interaction = client.interactions.create(
        model=GEMINI_MODEL_ID,
        input=[
            {
                "type": "image",
                "data": frame_b64,
                "mime_type": "image/png",
            },
            {
                "type": "text",
                "text": (
                    f'Look at this illustration for "{target_chapter["name"]}" '
                    f"and write a {video_duration_seconds}-second single-shot "
                    "story-continuation action prompt describing what "
                    "physically happens NEXT in the story.\n"
                    "- Start immediately with the characters in the image and "
                    "strong physical action verbs—do NOT re-describe the "
                    "painting style or static background.\n"
                    "- NEVER write a passive camera pan, slow zoom, tilt, or "
                    "parallax over frozen characters.\n"
                    f"- Choreograph 3 clear physical story beats ([0-{beat_1}s]"
                    f", [{beat_1}-{beat_2}s], [{beat_2}-"
                    f"{video_duration_seconds}s]) with expressive body/limb "
                    "movement, facial transitions, moving props or foliage, "
                    "and matching diegetic sound effects/foley.\n"
                    "- Include NO spoken dialogue, NO music, and NO on-screen "
                    "text."
                ),
            },
        ],
        previous_interaction_id=last_interaction.id,
        service_tier=service_tier,
    )
    last_interaction = motion_prompt_interaction
    video_prompt = motion_prompt_interaction.output_text.strip()
    display(safe_md(video_prompt))

    video_interaction = omni_client.interactions.create(
        model=OMNI_MODEL_ID,
        input=[
            {
                "type": "image",
                "uri": uploaded_frame.uri,
                "mime_type": uploaded_frame.mime_type or "image/png",
            },
            {
                "type": "text",
                "text": f"{video_prompt} Use this image as the starting frame.",
            },
        ],
        response_format={
            "type": "video",
            "delivery": "uri",
            "aspect_ratio": "9:16",
            "duration": f"{video_duration_seconds}s",
            "resolution": "720p",
        },
    )

    video_part = video_interaction.output_video
    if video_part and getattr(video_part, "uri", None):
        video_bytes = omni_client.files.download(file=video_part.uri)
    elif video_part and getattr(video_part, "data", None):
        video_bytes = base64.b64decode(video_part.data)
    else:
        raise RuntimeError("No video output returned from Omni model.")

    video_path = "chapter_video.mp4"
    with open(video_path, "wb") as video_file:
        video_file.write(video_bytes)

    display(Video(video_path, embed=True, width=360))

## 7/ Generate chapter music with Lyria 3.5

**[Lyria 3.5](https://deepmind.google/technologies/lyria/)** (`lyria-3.5`) composes high-fidelity songs and instrumental tracks from text prompts (`44.1 kHz` stereo MP3). Ask Gemini to write a cohesive musical direction for every chapter in the book, then slice to `max_chapter_images` in Python to compose each soundtrack using a scoped `v1beta` client (`lyria-3.5` is currently served on `v1beta`).

For timestamped song structures, vocals, and visual soundtracks, check out the [Lyria 3.5 quickstart](../quickstarts/Get_started_Lyria.ipynb).

In [ ]:
if not I_understand_this_is_a_paid_API:
    print(
        "Music generation is a paid feature. Set "
        "'I_understand_this_is_a_paid_API' to True above if you want to run it."
    )
else:
    from IPython.display import Audio

    lyria_client = genai.Client(
        api_key=GEMINI_API_KEY,
        http_options=types.HttpOptions(
            api_version="v1beta",
            retry_options=retry_options,
        ),
    )

    music_prompts_interaction = client.interactions.create(
        model=GEMINI_MODEL_ID,
        input=(
            "Create original instrumental music prompts for Lyria 3.5 for "
            "EVERY chapter or story in the book in chronological order using "
            "only musical genres, BPM, key signature, mood, and instruments. "
            "Do not mention book titles, character names, composers, or "
            "existing artists."
        ),
        previous_interaction_id=chapters_prompts_interaction.id,
        response_format={
            "type": "text",
            "mime_type": "application/json",
            "schema": {"type": "array", "items": Prompt.model_json_schema()},
        },
        service_tier=service_tier,
    )
    last_interaction = music_prompts_interaction
    chapters_music_prompts = json.loads(music_prompts_interaction.output_text)
    selected_music_prompts = chapters_music_prompts[:max_chapter_images]
    print(
        f"{len(chapters_music_prompts)} chapter music prompts ready "
        f"(composing {len(selected_music_prompts)})."
    )

    music_interaction = None
    generated_tracks = []
    for idx, chapter in enumerate(selected_music_prompts):
        display(safe_md(f"### Music for: {chapter['name']}"))
        display(safe_md(chapter["prompt"]))

        candidate_music = lyria_client.interactions.create(
            model=LYRIA_MODEL_ID,
            input=chapter["prompt"],
            response_modalities=["audio"],
        )

        music_audio = candidate_music.output_audio
        if music_audio and music_audio.data:
            music_interaction = candidate_music
            audio_bytes = base64.b64decode(music_audio.data)
            mime = (getattr(music_audio, "mime_type", "") or "").lower()
            is_mp3 = (
                "mpeg" in mime
                or "mp3" in mime
                or audio_bytes[:3] == b"ID3"
                or audio_bytes[:2] in (b"\xff\xfb", b"\xff\xf3", b"\xff\xf2")
            )
            ext = "mp3" if is_mp3 else "wav"
            track_file = f"chapter_music_{idx}.{ext}"
            with open(track_file, "wb") as f:
                f.write(audio_bytes)
            generated_tracks.append(track_file)
            display(Audio(track_file))

## 8/ Multi-character theatrical scene with Gemini 3.8 TTS

Dramatize a lively **three-character** scene from the book using **Gemini 3.8 TTS** (`gemini-3.8-flash-tts`, currently served on `v1beta`) across **two prebuilt voices** (`tts_voice_1` and `tts_voice_2`):
- Each character gets its own `CharacterVoiceProfile` with a distinct baseline vocal persona (`character_prompt`: pitch/register, tempo, cadence, and attitude).
- Even when **two characters share one voice slot**, contrasting `character_prompt` directions (for example, a slow, booming bass vs. a rapid-fire, cheeky voice) plus per-turn `turn_emotion` cues in `speech_metadata` and sparse inline vocal tags (`<laugh>`, `<sigh>`, `<whisper>`, `<short pause>`) keep every character distinct.

For custom Voice Design, Voice Replication, and larger ensembles, see the [TTS quickstart](../quickstarts/Get_started_TTS.ipynb).

In [ ]:
# @title Audio playback helpers { display-mode: "form" }
import contextlib
import wave
from IPython.display import Audio

file_index = 0


@contextlib.contextmanager
def wave_file(filename, channels=1, rate=24000, sample_width=2):
    with wave.open(filename, "wb") as wf:
        wf.setnchannels(channels)
        wf.setsampwidth(sample_width)
        wf.setframerate(rate)
        yield wf


def play_audio_blob_bytes(audio_bytes):
    global file_index
    file_index += 1
    fname = f"audio_{file_index}.wav"
    if audio_bytes.startswith(b"RIFF"):
        with open(fname, "wb") as f:
            f.write(audio_bytes)
    else:
        with wave_file(fname) as wav:
            wav.writeframes(audio_bytes)
    display(Audio(fname))


def play_audio(interaction):
    audio_part = interaction.output_audio
    if audio_part and audio_part.data:
        play_audio_blob_bytes(base64.b64decode(audio_part.data))
        return
    print("No audio found in interaction")

In [ ]:
tts_voice_1 = "Charon"  # @param ["Charon", "Kore", "Puck", "Aoede", "Fenrir", "Leda", "Zephyr", "Orus"] {"allow-input": true, "isTemplate": true}
tts_voice_2 = "Kore"  # @param ["Charon", "Kore", "Puck", "Aoede", "Fenrir", "Leda", "Zephyr", "Orus"] {"allow-input": true, "isTemplate": true}

if not I_understand_this_is_a_paid_API:
    print(
        "TTS is a paid feature. Set 'I_understand_this_is_a_paid_API' to "
        "True above if you want to use it."
    )
else:
    from typing import Literal
    from pydantic import Field

    class CharacterVoiceProfile(BaseModel):
        character_name: str = Field(
            description="Exact character name as used in turns"
        )
        voice_slot: Literal["Voice_1", "Voice_2"] = Field(
            description="Assigned prebuilt voice slot ('Voice_1' or 'Voice_2')"
        )
        character_prompt: str = Field(
            description=(
                "Distinct vocal persona (pitch/register, tempo, cadence, "
                "attitude). Characters sharing a voice_slot must have sharply "
                "contrasting registers, tempos, and vocal textures."
            )
        )

    class DialogTurn(BaseModel):
        character_name: str = Field(
            description=(
                "Character speaking this line (matching CharacterVoiceProfile)"
            )
        )
        turn_emotion: str = Field(
            description="Concise acting beat and emotional shift for this line"
        )
        text: str = Field(
            description=(
                "Verbatim spoken line only (no stage directions; optional "
                "sparse inline tags like <laugh>, <sigh>, <whisper>, "
                "<short pause>)"
            )
        )

    class SceneDialog(BaseModel):
        scene_title: str
        characters: list[CharacterVoiceProfile] = Field(min_length=1)
        turns: list[DialogTurn] = Field(min_length=1)

    dialog_interaction = client.interactions.create(
        model=GEMINI_MODEL_ID,
        input=[
            {
                "type": "text",
                "text": (
                    "Pick a funny, high-energy 3-character dialogue scene "
                    "from the book and adapt it into a punchy 7-to-9-turn "
                    "theatrical radio scene.\n"
                    "- Define exactly 3 CharacterVoiceProfile entries mapped "
                    "across ONLY 2 voice slots ('Voice_1' and 'Voice_2').\n"
                    "- Because two characters share one voice slot, give those "
                    "two sharply contrasting character_prompt directions (such "
                    "as a deep, slow, booming bass vs. a rapid-fire, "
                    "high-pitched voice) so listeners can tell all three apart "
                    "immediately.\n"
                    "- Keep dialogue faithful to the book's storytelling "
                    "diction (avoid modern slang), 1-2 snappy sentences per "
                    "turn, with at most one natural inline tag (<laugh>, "
                    "<sigh>, <whisper>, or <short pause>) where fitting."
                ),
            },
            {"type": "document", "uri": book.uri},
        ],
        response_format={
            "type": "text",
            "mime_type": "application/json",
            "schema": SceneDialog.model_json_schema(),
        },
        service_tier=service_tier,
    )

    scene = SceneDialog.model_validate_json(dialog_interaction.output_text)
    if not scene.characters or not scene.turns:
        raise RuntimeError("SceneDialog returned empty characters or turns.")

    voice_names = {"Voice_1": tts_voice_1, "Voice_2": tts_voice_2}
    esc_title = html.escape(scene.scene_title, quote=True)
    display(Markdown(f"### Theatrical Scene: {esc_title}"))

    profile_lookup = {
        prof.character_name.strip().lower(): prof for prof in scene.characters
    }
    for prof in scene.characters:
        esc_cname = html.escape(prof.character_name, quote=True)
        raw_vname = voice_names.get(prof.voice_slot, prof.voice_slot)
        esc_vname = html.escape(raw_vname, quote=True)
        esc_cprompt = html.escape(prof.character_prompt, quote=True)
        display(
            Markdown(
                f"- **{esc_cname}** *(voice `{esc_vname}` via "
                f"`{prof.voice_slot}`)*: {esc_cprompt}"
            )
        )

    fallback_profile = scene.characters[0]
    tts_input = []
    for turn in scene.turns:
        prof = profile_lookup.get(
            turn.character_name.strip().lower(), fallback_profile
        )
        esc_char = html.escape(turn.character_name, quote=True)
        raw_vname = voice_names.get(prof.voice_slot, prof.voice_slot)
        esc_vname = html.escape(raw_vname, quote=True)
        esc_emo = html.escape(turn.turn_emotion, quote=True)
        esc_txt = html.escape(turn.text, quote=True)
        display(
            Markdown(f"**{esc_char}** *({esc_vname} — {esc_emo})*: {esc_txt}")
        )
        tts_input.append(
            {
                "type": "text",
                "text": turn.text,
                "annotations": [
                    {
                        "type": "speech_metadata",
                        "speaker": prof.voice_slot,
                        "style": (
                            f"Character ({prof.character_name}): "
                            f"{prof.character_prompt}. "
                            f"Line delivery: {turn.turn_emotion}"
                        ),
                    }
                ],
            }
        )

    tts_client = genai.Client(
        api_key=GEMINI_API_KEY,
        http_options=types.HttpOptions(
            api_version="v1beta",
            retry_options=retry_options,
        ),
    )

    tts_interaction = tts_client.interactions.create(
        model=TTS_MODEL_ID,
        input=tts_input,
        response_format={"type": "audio"},
        generation_config={
            "speech_config": [
                {"speaker": "Voice_1", "voice": tts_voice_1},
                {"speaker": "Voice_2", "voice": tts_voice_2},
            ]
        },
    )

    play_audio(tts_interaction)

## 9/ Mix everything together

Combine the portrait illustration, the multi-speaker TTS narration (`24 kHz` WAV), and the **Lyria 3.5** background music (`44.1 kHz` stereo MP3) into a mixed storybook video exported at `44.1 kHz` / `320k` AAC so the stereo soundtrack retains full fidelity:

In [ ]:
if (
    not I_understand_this_is_a_paid_API
    or "tts_interaction" not in locals()
    or tts_interaction is None
):
    print(
        "TTS and music mixing require paid features. Set "
        "'I_understand_this_is_a_paid_API = True' and run Sections 7 and 8 "
        "first."
    )
else:
    %pip install -U -q "moviepy>=2.0.0"

    import os
    import shutil
    from IPython.display import Video
    from moviepy import AudioFileClip, CompositeAudioClip, ImageClip, afx

    print("Extracting TTS narration...")
    tts_audio = tts_interaction.output_audio
    if not tts_audio or not tts_audio.data:
        raise ValueError("No TTS audio found in tts_interaction.")
    tts_audio_bytes = base64.b64decode(tts_audio.data)

    if tts_audio_bytes.startswith(b"RIFF"):
        with open("tts.wav", "wb") as f:
            f.write(tts_audio_bytes)
    else:
        with wave.open("tts.wav", "wb") as wf:
            wf.setnchannels(1)
            wf.setsampwidth(2)
            wf.setframerate(24000)
            wf.writeframes(tts_audio_bytes)

    print("Saving chapter illustration...")
    if "last_chapter_image" in locals() and last_chapter_image is not None:
        last_chapter_image.save("illustration.png")
    elif (
        "generated_image" in locals()
        and generated_image is not None
        and generated_image.data
    ):
        with open("illustration.png", "wb") as f:
            f.write(base64.b64decode(generated_image.data))
    else:
        raise RuntimeError(
            "No chapter illustration available; run Section 5 first."
        )

    print("Extracting Lyria 3.5 background music...")
    music_file: str | None = None
    if "music_interaction" in locals() and music_interaction:
        music_audio = music_interaction.output_audio
        if music_audio and music_audio.data:
            music_bytes = base64.b64decode(music_audio.data)
            mime = (getattr(music_audio, "mime_type", "") or "").lower()
            if music_bytes.startswith(b"RIFF"):
                music_file = "music.wav"
                with open(music_file, "wb") as f:
                    f.write(music_bytes)
            elif (
                "mpeg" in mime
                or "mp3" in mime
                or music_bytes[:3] == b"ID3"
                or music_bytes[:2]
                in (b"\xff\xfb", b"\xff\xf3", b"\xff\xf2")
            ):
                music_file = "music.mp3"
                with open(music_file, "wb") as f:
                    f.write(music_bytes)
            else:
                music_file = "music.wav"
                with wave.open(music_file, "wb") as wf:
                    wf.setnchannels(1)
                    wf.setsampwidth(2)
                    wf.setframerate(48000)
                    wf.writeframes(music_bytes)

    if not music_file and "generated_tracks" in locals() and generated_tracks:
        src_track = generated_tracks[-1]
        _, track_ext = os.path.splitext(src_track)
        music_file = f"music{track_ext or '.mp3'}"
        shutil.copyfile(src_track, music_file)
    if not music_file:
        raise RuntimeError(
            "No Lyria 3.5 background music available; run Section 7 first."
        )

    print("Mixing narration and soundtrack with MoviePy...")
    tts_clip = AudioFileClip("tts.wav")
    raw_music_clip = None
    scaled_music_clip = None
    music_clip = None
    final_audio = None
    img_clip = None
    video = None
    try:
        raw_music_clip = AudioFileClip(music_file)
        scaled_music_clip = raw_music_clip.with_effects(
            [afx.MultiplyVolume(0.24)]
        )
        if scaled_music_clip.duration < tts_clip.duration:
            music_clip = scaled_music_clip.with_effects(
                [afx.AudioLoop(duration=tts_clip.duration)]
            )
        else:
            music_clip = scaled_music_clip.subclipped(0, tts_clip.duration)
        final_audio = CompositeAudioClip([tts_clip, music_clip])

        img_clip = ImageClip("illustration.png").with_duration(
            tts_clip.duration
        )
        video = img_clip.with_audio(final_audio)

        video_file = "chapter_mixed.mp4"
        video.write_videofile(
            video_file,
            fps=1,
            codec="libx264",
            audio_codec="aac",
            audio_fps=44100,
            audio_bitrate="320k",
            logger=None,
        )
    finally:
        for clip in (
            video,
            img_clip,
            final_audio,
            music_clip,
            scaled_music_clip,
            raw_music_clip,
            tts_clip,
        ):
            if clip is not None:
                clip.close()

    print("Done! Displaying mixed storybook video:")
    display(Video(video_file, embed=True, width=360))

# With an audiobook: The Adventures of Chatterer the Red Squirrel

Because Gemini is natively multimodal, you can run the exact same illustration workflow directly from an **audio recording**—without transcribing it first. This example uses the public-domain audiobook *[The Adventures of Chatterer the Red Squirrel](https://librivox.org/the-adventures-of-chatterer-the-red-squirel-by-thornton-w-burgess/)* by Thornton W. Burgess from [LibriVox](https://librivox.org/).

## 1/ Get the audiobook and merge its chapters

Download the first 4 chapter MP3s directly from Internet Archive and merge them into a single `audiobook.mp3` file so you only upload one audio file. Feel free to add more chapter URLs to illustrate the full book.

In [ ]:
%pip install -U -q pydub audioop-lts

from pydub import AudioSegment

base_url = (
    "https://archive.org/download/chatterertheredsquirrel_1307_librivox/"
    "adventuresofchatterertheredsquirrel_{:02d}_burgess_64kb.mp3"
)
chapter_urls = [base_url.format(i) for i in range(1, 5)]

combined_audio = AudioSegment.empty()
total_bytes = 0
for idx, chapter_url in enumerate(chapter_urls, start=1):
    resp = requests.get(
        chapter_url,
        headers={"User-Agent": "Mozilla/5.0"},
        timeout=60,
    )
    resp.raise_for_status()
    total_bytes += len(resp.content)
    combined_audio += AudioSegment.from_mp3(io.BytesIO(resp.content))
    print(
        f"Downloaded chapter {idx}/{len(chapter_urls)} "
        f"({len(resp.content):,} bytes)"
    )

combined_audio.export("audiobook.mp3", format="mp3")
print(
    f"Merged {len(chapter_urls)} chapters into audiobook.mp3 "
    f"({total_bytes:,} source bytes)"
)

Upload `audiobook.mp3` with the Files API and wait until `ACTIVE`:

In [ ]:
audiobook = wait_for_file_active(client.files.upload(file="audiobook.mp3"))
print(f"Audiobook ready: {audiobook.name}")

## 2/ Start the chat

Chain interactions via `previous_interaction_id` so Gemini listens to the uploaded audiobook once and reuses that audio context across every subsequent turn, using [structured outputs](https://ai.google.dev/gemini-api/docs/structured-output?lang=python#generate-json) for prompt extraction:

In [ ]:
# Re-run this cell if you want to start a fresh audiobook session
audiobook_interaction = client.interactions.create(
    model=GEMINI_MODEL_ID,
    input=[
        {
            "type": "text",
            "text": (
                "Here is an audiobook to illustrate using Nano Banana. "
                "Do not say anything for now; instructions will follow."
            ),
        },
        {
            "type": "audio",
            "uri": audiobook.uri,
            "mime_type": audiobook.mime_type or "audio/mpeg",
        },
    ],
    service_tier=service_tier,
)
last_interaction = audiobook_interaction

## 3/ Define a style

Specify a custom visual style—here a futuristic sci-fi aesthetic—or clear the field to let Gemini propose one from the audio narrative:

In [ ]:
style = "futuristic, science fiction, utopia, saturated, neon lights"  # @param {"type": "string", "placeholder": "Write your own style or leave empty to let Gemini generate one"}

if style == "":
    interaction = client.interactions.create(
        model=GEMINI_MODEL_ID,
        input=(
            "Define an art style that fits this audiobook story. "
            "Return only the concise style prompt to append to future prompts."
        ),
        previous_interaction_id=last_interaction.id,
        service_tier=service_tier,
    )
    last_interaction = interaction
    style = interaction.output_text
else:
    interaction = client.interactions.create(
        model=GEMINI_MODEL_ID,
        input=(
            f'The art style will be: "{style}". Keep that in mind when '
            "generating future prompts. Wait quietly for the next instructions."
        ),
        previous_interaction_id=last_interaction.id,
        service_tier=service_tier,
    )
    last_interaction = interaction

display(Markdown("### Style"))
print(style)

style = f'Follow this style: "{style}"'

## 4/ Generate portraits of the main characters

Ask Gemini to extract character descriptions directly from the spoken narration and generate portrait illustrations:

In [ ]:
interaction = client.interactions.create(
    model=GEMINI_MODEL_ID,
    input=(
        "Describe the main recurring animal or adult characters heard across "
        "the entire audiobook (do not include human children) and write a "
        "detailed portrait prompt for each one so Nano Banana can illustrate "
        "them."
    ),
    previous_interaction_id=last_interaction.id,
    response_format={
        "type": "text",
        "mime_type": "application/json",
        "schema": {"type": "array", "items": Prompt.model_json_schema()},
    },
    service_tier=service_tier,
)
last_interaction = interaction

all_audiobook_characters = json.loads(interaction.output_text)
print(f"{len(all_audiobook_characters)} audiobook character prompts ready.")
print(json.dumps(all_audiobook_characters, indent=2))

In [ ]:
audiobook_characters = all_audiobook_characters[:max_character_images]
print(
    f"Rendering {len(audiobook_characters)} of "
    f"{len(all_audiobook_characters)} audiobook character portraits..."
)
last_image_interaction = None

for character in audiobook_characters:
    display(safe_md(f"### {character['name']}"))
    display(safe_md(character["prompt"]))

    turn_kwargs = {
        "model": IMAGE_MODEL_ID,
        "input": (
            "Illustrate a character portrait for this audiobook story.\n"
            f"{style}\n"
            f"Character ({character['name']}): {character['prompt']}"
        ),
        "system_instruction": system_instructions,
        "response_format": {"type": "image", "aspect_ratio": "9:16"},
        "service_tier": service_tier,
    }
    if last_image_interaction is not None:
        turn_kwargs["previous_interaction_id"] = last_image_interaction.id

    interaction = client.interactions.create(**turn_kwargs)
    last_image_interaction = interaction

    generated_image = interaction.output_image
    if generated_image and generated_image.data:
        display_generated_image(generated_image)

## 5/ Illustrate the chapters of the audiobook

Extract scene prompts for every spoken chapter in the audiobook (`all_audiobook_chapters`), then slice to `max_chapter_images` in Python to render consistent portrait illustrations:

In [ ]:
interaction = client.interactions.create(
    model=GEMINI_MODEL_ID,
    input=(
        "Write one detailed single-scene illustration prompt for EVERY "
        "chapter or story heard in the audiobook in chronological order, "
        "reusing the character descriptions where they appear."
    ),
    previous_interaction_id=last_interaction.id,
    response_format={
        "type": "text",
        "mime_type": "application/json",
        "schema": {"type": "array", "items": Prompt.model_json_schema()},
    },
    service_tier=service_tier,
)
last_interaction = interaction

all_audiobook_chapters = json.loads(interaction.output_text)
print(f"{len(all_audiobook_chapters)} audiobook chapters ready.")
print(json.dumps(all_audiobook_chapters, indent=2))

In [ ]:
audiobook_chapters = all_audiobook_chapters[:max_chapter_images]
print(
    f"Illustrating {len(audiobook_chapters)} of "
    f"{len(all_audiobook_chapters)} audiobook chapters..."
)

for chapter in audiobook_chapters:
    display(safe_md(f"### {chapter['name']}"))
    display(safe_md(chapter["prompt"]))

    turn_kwargs = {
        "model": IMAGE_MODEL_ID,
        "input": (
            f"Create an illustration for {chapter['name']} keeping the "
            "previously generated characters consistent while adapting their "
            f"poses:\n{style}\n{chapter['prompt']}"
        ),
        "system_instruction": system_instructions,
        "response_format": {"type": "image", "aspect_ratio": "9:16"},
        "service_tier": service_tier,
    }
    if last_image_interaction is not None:
        turn_kwargs["previous_interaction_id"] = last_image_interaction.id

    interaction = client.interactions.create(**turn_kwargs)
    last_image_interaction = interaction

    generated_image = interaction.output_image
    if generated_image and generated_image.data:
        display_generated_image(generated_image)

# Next steps

**Useful documentation references:**

Check the documentation and quickstarts for more details about the features used in this notebook:
- [Interactions API](https://ai.google.dev/gemini-api/docs/interactions): Chain stateful multimodal turns with `previous_interaction_id`.
- [Priority inference](https://ai.google.dev/gemini-api/docs/priority-inference) & [Inference tiers quickstart](../quickstarts/Inference_tiers.ipynb): Route interactive requests to Priority queues or background chains to Flex queues.
- [Image generation](https://ai.google.dev/gemini-api/docs/image-generation) & [Nano Banana quickstart](../quickstarts/Get_Started_Nano_Banana.ipynb): Generate and edit multi-reference illustrations with `gemini-nano-banana-2.1` and `gemini-3.1-flash-lite-image`.
- [Video generation and editing](https://ai.google.dev/gemini-api/docs/omni) & [Omni quickstart](../quickstarts/Get_started_Omni.ipynb): Animate frames and edit videos conversationally with `gemini-omni-1.2-flash`.
- [Music generation](https://ai.google.dev/gemini-api/docs/music-generation) & [Lyria 3.5 quickstart](../quickstarts/Get_started_Lyria.ipynb): Compose songs and instrumental tracks with `lyria-3.5`.
- [Speech generation](https://ai.google.dev/gemini-api/docs/speech-generation) & [Voice design](https://ai.google.dev/gemini-api/docs/voice-design) ([TTS quickstart](../quickstarts/Get_started_TTS.ipynb)): Synthesize expressive multi-speaker dialogue with `gemini-3.8-flash-tts`.

**Related examples:**

Check out these other end-to-end application examples:
- [Autonomous video production with Gemini Omni and agents](./Autonomous_video_production_with_omni_and_agents.ipynb)
- [Story writing with prompt chaining](./Story_Writing_with_Prompt_Chaining.ipynb)
- [Plotting and mapping using the Live API](./LiveAPI_plotting_and_mapping.ipynb)

**Continue your discovery of the Gemini API:**

Browse the [Quickstarts folder](https://github.com/google-gemini/cookbook/tree/main/quickstarts/) in the cookbook to learn more about all Gemini API capabilities.